<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/03_AI_Engineer/advanced/04_advanced_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Retrieval & GraphRAG

Push retrieval quality to the limit — knowledge graphs, ColBERT late interaction, contextual compression, and adaptive RAG routing.

**Topics:** Knowledge graph construction, graph-based retrieval, ColBERT late interaction, SPLADE sparse retrieval, contextual compression, adaptive RAG

## 1. Knowledge Graph Construction from Text

In [ ]:
import json
import os
from openai import OpenAI
from dataclasses import dataclass, field

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

@dataclass
class KGTriple:
    subject: str
    predicate: str
    obj: str
    source: str = ''
    confidence: float = 1.0

@dataclass
class KnowledgeGraph:
    nodes: set[str] = field(default_factory=set)
    edges: list[KGTriple] = field(default_factory=list)
    node_metadata: dict[str, dict] = field(default_factory=dict)
    
    def add_triple(self, triple: KGTriple):
        self.nodes.add(triple.subject)
        self.nodes.add(triple.obj)
        self.edges.append(triple)
    
    def get_neighbors(self, node: str) -> list[tuple[str, str, str]]:
        """Get all (predicate, object) pairs for a given subject."""
        return [(e.predicate, e.obj) for e in self.edges if e.subject.lower() == node.lower()]
    
    def stats(self) -> dict:
        return {'nodes': len(self.nodes), 'edges': len(self.edges), 'avg_degree': len(self.edges) / max(len(self.nodes), 1)}

EXTRACT_PROMPT = """Extract knowledge graph triplets from this text.
Each triplet: (subject, predicate, object)
- subject and object should be concrete entities
- predicate should be a verb phrase
- Be specific and factual

Text: {text}

Return JSON: {"triplets": [["subject", "predicate", "object"], ...]}"""

def extract_triplets(text: str, source: str = '') -> list[KGTriple]:
    result = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': EXTRACT_PROMPT.format(text=text)}],
        response_format={'type': 'json_object'},
    )
    data = json.loads(result.choices[0].message.content)
    return [KGTriple(subject=t[0], predicate=t[1], obj=t[2], source=source) for t in data.get('triplets', [])]

# Simulated extracted triplets
kg = KnowledgeGraph()
sample_triplets = [
    KGTriple('BERT', 'uses', 'bidirectional attention', 'bert_paper', 0.99),
    KGTriple('BERT', 'was_trained_on', 'Wikipedia', 'bert_paper', 0.99),
    KGTriple('GPT-3', 'has_parameter_count', '175 billion', 'gpt3_paper', 0.99),
    KGTriple('attention mechanism', 'introduced_by', 'Vaswani et al.', 'attention_paper', 0.99),
    KGTriple('transformer', 'uses', 'attention mechanism', 'attention_paper', 0.99),
    KGTriple('BERT', 'is_a', 'transformer', 'bert_paper', 0.99),
    KGTriple('GPT-3', 'is_a', 'transformer', 'gpt3_paper', 0.99),
]
for t in sample_triplets:
    kg.add_triple(t)

print(f'Knowledge Graph: {kg.stats()}')
print(f'\nBERT neighbors:')
for pred, obj in kg.get_neighbors('BERT'):
    print(f'  BERT -{pred}→ {obj}')

## 2. Graph-Based Retrieval — Entity-Centric Search

In [ ]:
from collections import deque

def extract_entities_from_query(query: str) -> list[str]:
    """Identify entities in the query for graph traversal."""
    result = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': f'List the named entities in this query as JSON list. Query: {query}'}],
        response_format={'type': 'json_object'},
    )
    data = json.loads(result.choices[0].message.content)
    return data.get('entities', [])

def graph_retrieve(
    query: str,
    kg: KnowledgeGraph,
    max_hops: int = 2,
    max_nodes: int = 20,
) -> list[KGTriple]:
    """BFS from query entities through the knowledge graph.
    
    Why graph retrieval excels over vector search:
    - Multi-hop reasoning: BERT → is_a → transformer → uses → attention
    - Precise entity matching (no embedding drift)
    - Relationship-aware context
    """
    seed_entities = extract_entities_from_query(query)
    
    visited_nodes: set[str] = set()
    queue = deque([(entity, 0) for entity in seed_entities])
    retrieved_triples: list[KGTriple] = []
    
    while queue and len(visited_nodes) < max_nodes:
        current_node, hops = queue.popleft()
        if current_node in visited_nodes or hops > max_hops:
            continue
        visited_nodes.add(current_node)
        
        # Get all edges from this node
        node_triples = [e for e in kg.edges if e.subject.lower() == current_node.lower()]
        retrieved_triples.extend(node_triples)
        
        # Add neighbors to queue for next hop
        for triple in node_triples:
            if triple.obj not in visited_nodes:
                queue.append((triple.obj, hops + 1))
    
    return retrieved_triples

# Demo graph retrieval
query = 'How does BERT relate to transformers and attention?'
seed_entities_simulated = ['BERT', 'transformers', 'attention']

print(f'Query: "{query}"')
print(f'Seed entities: {seed_entities_simulated}')
print()
print('Graph traversal (2-hop BFS):')
print('  Hop 0: BERT, transformers, attention')
print('  Hop 1: bidirectional attention, Wikipedia, Vaswani et al., 175B...')
print('  Hop 2: attention mechanism papers, related entities...')
print()
print('Retrieved triples for context:')
retrieved = [t for t in kg.edges if 'BERT' in t.subject or 'transformer' in t.subject.lower()]
for t in retrieved:
    print(f'  ({t.subject}) -{t.predicate}→ ({t.obj})')

## 3. ColBERT Late Interaction Scoring

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

def colbert_score(query_embeddings: np.ndarray, doc_embeddings: np.ndarray) -> float:
    """MaxSim: for each query token, find max similarity with any document token.
    
    ColBERT insight: don't average token embeddings!
    Instead: score(Q, D) = Σ max_{d∈D} (q · d) for each q∈Q
    
    This preserves token-level semantics:
    'BERT' query token matches 'BERT' doc token exactly
    'attention' query token matches 'self-attention' doc token closely
    """
    # query_embeddings: (n_query_tokens, dim)
    # doc_embeddings: (n_doc_tokens, dim)
    
    # Normalize
    q_norm = query_embeddings / np.linalg.norm(query_embeddings, axis=1, keepdims=True)
    d_norm = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    
    # Similarity matrix: (n_query, n_doc)
    sim_matrix = q_norm @ d_norm.T
    
    # MaxSim: for each query token, max over doc tokens
    max_sims = sim_matrix.max(axis=1)  # (n_query,)
    
    return float(max_sims.sum())

# Compare bi-encoder vs ColBERT
np.random.seed(42)

def simulate_query_doc_scores():
    dim = 128
    query_tokens = np.random.randn(8, dim)   # 8 query tokens
    doc_a_tokens = np.random.randn(20, dim)  # 20 doc tokens
    doc_b_tokens = np.random.randn(20, dim)
    
    # Make doc_a more relevant (inject matching token)
    doc_a_tokens[5] = query_tokens[2] + np.random.randn(dim) * 0.1
    
    colbert_a = colbert_score(query_tokens, doc_a_tokens)
    colbert_b = colbert_score(query_tokens, doc_b_tokens)
    
    # Bi-encoder (mean pooling): loses token-level matching
    bi_q = query_tokens.mean(axis=0)
    bi_a = doc_a_tokens.mean(axis=0)
    bi_b = doc_b_tokens.mean(axis=0)
    bi_score_a = float(bi_q @ bi_a / (np.linalg.norm(bi_q) * np.linalg.norm(bi_a)))
    bi_score_b = float(bi_q @ bi_b / (np.linalg.norm(bi_q) * np.linalg.norm(bi_b)))
    
    return colbert_a, colbert_b, bi_score_a, bi_score_b

ca, cb, ba, bb = simulate_query_doc_scores()
print('ColBERT MaxSim vs Bi-encoder comparison:')
print(f'  ColBERT:    doc_A={ca:.2f}, doc_B={cb:.2f} → ranks A higher: {ca > cb}')
print(f'  Bi-encoder: doc_A={ba:.4f}, doc_B={bb:.4f} → ranks A higher: {ba > bb}')
print()
print('ColBERT advantage: injected matching token in doc_A is captured by MaxSim')
print('Bi-encoder misses it (diluted by mean pooling over all tokens)')

## 4. Contextual Compression Retriever

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class CompressedChunk:
    original_text: str
    compressed_text: str
    source: str
    compression_ratio: float
    relevance_score: float

COMPRESS_PROMPT = """Given the following passage and a user query, extract ONLY the sentences and phrases
that are directly relevant to answering the query. Remove all irrelevant content.
If nothing is relevant, output: IRRELEVANT

Query: {query}

Passage: {passage}

Relevant excerpts only:"""

def contextual_compress(
    query: str,
    chunks: list[str],
    sources: list[str],
    max_tokens_per_chunk: int = 200,
) -> list[CompressedChunk]:
    """Extract only query-relevant sentences from each retrieved chunk.
    
    Problem with naive RAG: retrieved chunks often contain 80% irrelevant text.
    Compression: keep only the relevant 20%, fitting 5x more information in context.
    """
    compressed = []
    for chunk, source in zip(chunks, sources):
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': COMPRESS_PROMPT.format(query=query, passage=chunk)}],
            max_tokens=max_tokens_per_chunk,
        )
        compressed_text = response.choices[0].message.content.strip()
        
        if compressed_text == 'IRRELEVANT':
            continue  # Drop completely irrelevant chunks
        
        ratio = len(compressed_text) / len(chunk)
        compressed.append(CompressedChunk(
            original_text=chunk, compressed_text=compressed_text,
            source=source, compression_ratio=ratio, relevance_score=1.0 - ratio,
        ))
    
    return compressed

# Compression impact analysis
print('Contextual compression impact:')
print()
scenario = {
    'chunks_retrieved': 10,
    'avg_chunk_tokens': 512,
    'compression_ratio': 0.2,
    'irrelevant_chunks_removed': 3,
}
original_tokens = scenario['chunks_retrieved'] * scenario['avg_chunk_tokens']
after_removal = (scenario['chunks_retrieved'] - scenario['irrelevant_chunks_removed']) * scenario['avg_chunk_tokens']
after_compression = after_removal * scenario['compression_ratio']

print(f'  Original (10 chunks × 512 tokens): {original_tokens:,} tokens')
print(f'  After removing irrelevant:          {after_removal:,} tokens')
print(f'  After 5x compression:               {after_compression:.0f} tokens')
print(f'  Space saved: {(1 - after_compression/original_tokens)*100:.0f}%')
print()
print('Benefit: fit 5x more diverse sources in the same context window')

## 5. Adaptive RAG — Route Queries to Right Strategy

In [ ]:
from enum import Enum
from dataclasses import dataclass
from typing import Callable

class RetrievalStrategy(Enum):
    DIRECT_LLM = 'direct'           # No retrieval — LLM knows it
    VECTOR_SEARCH = 'vector'        # Standard dense retrieval
    HYBRID_SEARCH = 'hybrid'        # BM25 + dense
    GRAPH_SEARCH = 'graph'          # Knowledge graph traversal
    MULTI_QUERY = 'multi_query'     # Generate multiple queries
    HYDE = 'hyde'                   # Hypothetical document expansion
    RERANK = 'rerank'               # Retrieve + cross-encoder rerank

ROUTER_PROMPT = """Classify this query to determine the best retrieval strategy.

Strategies:
- direct: LLM knows the answer from training data (factual, common knowledge)
- vector: Standard semantic search (most queries)
- hybrid: Query has specific keywords + semantic meaning
- graph: Query involves relationships between named entities
- multi_query: Ambiguous or multi-faceted query
- hyde: Highly abstract or conceptual query
- rerank: High-stakes query requiring maximum precision

Query: {query}

Return JSON: {"strategy": "...", "reasoning": "..."}"""

@dataclass
class AdaptiveRAG:
    retrievers: dict[RetrievalStrategy, Callable]
    router_model: str = 'gpt-4o-mini'
    
    def retrieve(self, query: str) -> tuple[list, RetrievalStrategy]:
        # Step 1: Classify query → choose strategy
        response = client.chat.completions.create(
            model=self.router_model,
            messages=[{'role': 'user', 'content': ROUTER_PROMPT.format(query=query)}],
            response_format={'type': 'json_object'},
        )
        routing = json.loads(response.choices[0].message.content)
        strategy = RetrievalStrategy(routing['strategy'])
        
        # Step 2: Execute the chosen strategy
        retriever = self.retrievers.get(strategy, self.retrievers[RetrievalStrategy.VECTOR_SEARCH])
        results = retriever(query)
        return results, strategy

# Routing examples
routing_examples = [
    ('What is the capital of France?', RetrievalStrategy.DIRECT_LLM, 'Common knowledge, no retrieval needed'),
    ('Find all mentions of "CUDA out of memory"', RetrievalStrategy.HYBRID_SEARCH, 'Exact error string + semantic context'),
    ('How does BERT relate to GPT and transformers?', RetrievalStrategy.GRAPH_SEARCH, 'Entity relationships'),
    ('Best practices for production RAG', RetrievalStrategy.MULTI_QUERY, 'Broad topic, multiple angles'),
    ('What is the philosophical implication of emergence in LLMs?', RetrievalStrategy.HYDE, 'Abstract, needs doc-like expansion'),
    ('What is the exact accuracy of GPT-4 on MMLU benchmark?', RetrievalStrategy.RERANK, 'Precise fact, needs best matching'),
]

print('Adaptive RAG routing decisions:')
print(f'{"Query":<50} {"Strategy":<20} {"Reason"}')
print('-' * 100)
for query, strategy, reason in routing_examples:
    print(f'{query[:49]:<50} {strategy.value:<20} {reason}')
print()
print('Key benefit: 1 routing call (~$0.0002) can save 10x retrieval cost')
print('DIRECT_LLM paths: no embedding, no vector search, no API call to retriever!')

## Exercises

1. **GraphRAG benchmark:** Build a GraphRAG pipeline (extract KG → index → query). Benchmark it against pure vector RAG on 30 multi-hop reasoning questions (e.g., "Who trained the model that inspired BERT?"). Measure accuracy and latency for each approach.

2. **Adaptive router evaluation:** Create a labeled dataset of 50 queries with ground-truth optimal retrieval strategies. Train a small classifier (logistic regression on query embeddings) to predict the strategy without an LLM call. Measure accuracy and latency vs LLM router.

3. **ColBERT index:** Implement a ColBERT index that stores per-token embeddings for all documents. Build an efficient MaxSim search that avoids comparing all (query_token, doc_token) pairs — use an inverted index on token embeddings to prune candidates.